# Mage Iris ML Pipeline
End-to-end Machine Learning Pipeline built with Mage for the 'From Git to Gold' Data Engineering Assignment.

In [1]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn

import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help


Usage: mage [OPTIONS] COMMAND [ARGS]...

Commands:
  init   Initialize a new project.
  run    Run a pipeline.
  start  Start Mage web server.


In [2]:
!mage init iris_project
!ls iris_project


Initializing iris_project...
data_exporters  data_loaders  extensions  io_config.yaml  metadata.yaml  pipelines  transformers


In [3]:
%%writefile iris_project/data_loaders/load_iris.py
import pandas as pd
from sklearn.datasets import load_iris

if 'data_loader' not in globals():
    from mage_ai.data_preparation.decorators import data_loader
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@data_loader
def load_data(*args, **kwargs):
    iris = load_iris(as_frame=True)
    df = iris.frame
    print('Rows loaded:', len(df))
    return df

@test
def test_output(output, *args):
    assert len(output) == 150, 'Iris must have 150 rows'


Writing iris_project/data_loaders/load_iris.py


In [4]:
%%writefile iris_project/transformers/prepare_data.py
import pandas as pd
from sklearn.model_selection import train_test_split

if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def transform(df, *args, **kwargs):
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df['split'] = 'train'
    test_df['split'] = 'test'
    result = pd.concat([train_df, test_df])
    print('Train rows:', len(train_df), '| Test rows:', len(test_df))
    return result

@test
def test_output(output, *args):
    assert 'split' in output.columns, 'split column missing'


Writing iris_project/transformers/prepare_data.py


In [5]:
%%writefile iris_project/transformers/train_model.py
import joblib
from sklearn.tree import DecisionTreeClassifier

if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

FEATURES = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

@transformer
def train(df, *args, **kwargs):
    max_depth = int(kwargs.get('max_depth', 3))
    train_df = df[df['split'] == 'train']
    test_df = df[df['split'] == 'test'].copy()
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(train_df[FEATURES], train_df['target'])
    test_df['predicted'] = model.predict(test_df[FEATURES])
    accuracy = (test_df['predicted'] == test_df['target']).mean()
    print('max_depth =', max_depth, '| Test accuracy =', round(accuracy, 3))
    joblib.dump(model, 'iris_model.pkl')
    return test_df

@test
def test_accuracy(output, *args):
    accuracy = (output['predicted'] == output['target']).mean()
    assert accuracy >= 0.8, 'Model accuracy is below 80%'


Writing iris_project/transformers/train_model.py


In [6]:
%%writefile iris_project/data_exporters/save_results.py
if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter

@data_exporter
def export_data(df, *args, **kwargs):
    df.to_csv('iris_predictions.csv', index=False)
    print('Saved', len(df), 'predictions to iris_predictions.csv')


Writing iris_project/data_exporters/save_results.py


In [7]:
!mkdir -p iris_project/pipelines/iris_classifier
!touch iris_project/pipelines/iris_classifier/__init__.py


In [8]:
%%writefile iris_project/pipelines/iris_classifier/metadata.yaml
blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3


Writing iris_project/pipelines/iris_classifier/metadata.yaml


## C1: Baseline Pipeline Execution with Tests

In [9]:
!mage run iris_project iris_classifier --test


Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
Pipeline run completed.


In [10]:
import pandas as pd
pd.read_csv('iris_predictions.csv').head(10)


   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target split  predicted
0                6.1               2.8                4.7               1.2       1  test          1
1                5.7               3.8                1.7               0.3       0  test          0
2                7.7               2.6                6.9               2.3       2  test          2
3                6.0               2.2                5.0               1.5       1  test          1
4                6.8               2.8                4.8               1.4       1  test          1
5                5.4               3.4                1.5               0.4       0  test          0
6                5.6               2.9                3.6               1.3       1  test          1
7                6.9               3.1                5.1               2.3       2  test          2
8                6.2               2.2                4.5               1.5       1  test  

## C2: Quality Gate Demonstration Across max_depth 1 to 5

In [11]:
for depth in [1, 2, 3, 4, 5]:
    print(f'=== Running with max_depth = {depth} ===')
    !rm -f iris_predictions.csv
    !mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": '"$depth"'}'
    import os
    print(f'max_depth {depth} -> iris_predictions.csv created? {os.path.exists("iris_predictions.csv")}\n')


=== Running with max_depth = 1 ===
Rows loaded: 150
1/1 tests passed.
Train rows: 120 | Test rows: 30
1/1 tests passed.
max_depth = 1 | Test accuracy = 0.633
FAIL: test_accuracy (block: train_model)
AssertionError: Model accuracy is below 80%
max_depth 1 -> iris_predictions.csv created? False

=== Running with max_depth = 2 ===
Rows loaded: 150
1/1 tests passed.
Train rows: 120 | Test rows: 30
1/1 tests passed.
max_depth = 2 | Test accuracy = 0.967
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
max_depth 2 -> iris_predictions.csv created? True

=== Running with max_depth = 3 ===
Rows loaded: 150
1/1 tests passed.
Train rows: 120 | Test rows: 30
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
max_depth 3 -> iris_predictions.csv created? True

=== Running with max_depth = 4 ===
Rows loaded: 150
1/1 tests passed.
Train rows: 120 | Test rows: 30
1/1 tests passed.
max_depth = 4 | Test accuracy = 1.0
1/1 tes

## C3: Evaluation Branch with evaluate_model

In [12]:
%%writefile iris_project/transformers/evaluate_model.py
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer

@transformer
def evaluate(df, *args, **kwargs):
    y_true = df['target']
    y_pred = df['predicted']

    print('Confusion matrix:')
    print(pd.crosstab(y_true, y_pred, rownames=['actual'], colnames=['predicted']))

    metrics = pd.DataFrame({
        'metric': ['accuracy', 'precision', 'recall', 'f1_score'],
        'value': [
            round(accuracy_score(y_true, y_pred), 3),
            round(precision_score(y_true, y_pred, average='macro'), 3),
            round(recall_score(y_true, y_pred, average='macro'), 3),
            round(f1_score(y_true, y_pred, average='macro'), 3),
        ]
    })
    print(metrics.to_string(index=False))
    return metrics


Writing iris_project/transformers/evaluate_model.py


## C4: Custom Block check_data with Pre-Transformation Quality Gate

In [13]:
%%writefile iris_project/transformers/check_data.py
if 'transformer' not in globals():
    from mage_ai.data_preparation.decorators import transformer
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@transformer
def check(df, *args, **kwargs):
    missing = int(df.isna().sum().sum())
    classes = sorted(df['target'].unique().tolist())
    print('Missing values:', missing, '| Classes:', classes)
    return df

@test
def test_no_missing_and_three_classes(output, *args):
    assert output.isna().sum().sum() == 0, 'the data has missing values'
    assert output['target'].nunique() == 3, 'expected exactly 3 classes'


Writing iris_project/transformers/check_data.py


In [14]:
%%writefile iris_project/pipelines/iris_classifier/metadata.yaml
blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [check_data]
- name: check_data
  uuid: check_data
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [check_data]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3


Writing iris_project/pipelines/iris_classifier/metadata.yaml


In [15]:
!mage run iris_project iris_classifier --test


Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Missing values: 0 | Classes: [0, 1, 2]
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Saved 30 predictions to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
Pipeline run completed.
